# Pilot: Uji Coba Pipeline Training

Sebelum melatih model dengan data penuh (sekitar 800 ribu data), seluruh proses training dicoba dulu dengan data kecil. Tujuannya bukan untuk melihat model mana yang paling bagus, tetapi untuk memastikan semua bagian berjalan tanpa error, yaitu memuat data, melatih model, menilai di data validasi, menebak data test, mengukur efisiensi, dan menyimpan hasil.

Pilot ini juga dipakai untuk memperkirakan berapa lama training dengan data penuh nanti.

Yang dicoba ada 6 kombinasi, yaitu 3 jenis input (comment saja, early fusion, late fusion) untuk 2 model (RoBERTa dan DistilBERT). Semua memakai learning rate 3e-5.

## 1. Setup

Kode untuk training (memuat data, model, proses belajar, dan pencatatan hasil) ditulis di file `src/training.py`. Notebook ini cukup memanggil file tersebut, jadi kode yang diuji di sini sama dengan kode yang nanti dipakai untuk training data penuh.

Pengaturan pilot:
- data latih 2.000 (dari `train_pilot`),
- data validasi 1.000 dan data test 1.000, diambil seimbang antara sarkas dan bukan sarkas,
- 2 epoch.

Pilot sengaja dibuat 2 epoch. Epoch pertama biasanya lebih lambat karena GPU masih "pemanasan", jadi kecepatan dari epoch kedua lebih bisa dipakai untuk memperkirakan waktu training sebenarnya.

In [1]:
import sys, importlib
from pathlib import Path

import pandas as pd
import torch
import transformers

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "src"))
import training as T
importlib.reload(T)                       # kalau training.py diubah, cukup jalankan ulang sel ini
transformers.logging.set_verbosity_error()

DATA_DIR = ROOT / "data/tokenized_full"
OUT_DIR = ROOT / "data/training_rtx5060ti/pilot"
PILOT = dict(train_split="train_pilot", val_limit=1000, test_limit=1000, epochs=2)

free, total = torch.cuda.mem_get_info()
print("GPU:", torch.cuda.get_device_name(), f"| VRAM terpakai {(total - free) / 1e9:.1f} / {total / 1e9:.1f} GB")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| BF16:", torch.cuda.is_bf16_supported())

d:\Falah\TA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 5060 Ti | VRAM terpakai 1.2 / 17.1 GB
torch 2.11.0+cu128 | transformers 5.17.0 | BF16: True


#### Insight
GPU yang dipakai adalah NVIDIA GeForce RTX 5060 Ti. Sebelum training, memori GPU yang terpakai baru 1,2 GB dari 17,1 GB, dan itu dipakai untuk tampilan layar. GPU ini juga mendukung BF16, yaitu cara menghitung dengan angka yang lebih ringkas supaya training lebih cepat dan hemat memori.

Peringatan `TqdmWarning` di atas tidak berpengaruh apa-apa, hanya soal tampilan progress bar.

## 2. Daftar Run

Setiap kombinasi input dan model disebut satu run. Nama run berisi putaran, jenis input, model, learning rate, dan seed. Contohnya `R0_early_roberta_lr3e-05_s42` artinya putaran 0 (pilot), input early fusion, model RoBERTa, learning rate 3e-5, seed 42.

In [2]:
queue = T.build_queue([0])
T.run_status(queue, OUT_DIR)

,run_id,round,input,model,lr,seed,status
0,R0_comment_roberta_lr3e-05_s42,0,comment,roberta,0.00003,42,belum
1,R0_comment_distilbert_lr3e-05_s42,0,comment,distilbert,0.00003,42,belum
2,R0_early_roberta_lr3e-05_s42,0,early,roberta,0.00003,42,belum
3,R0_early_distilbert_lr3e-05_s42,0,early,distilbert,0.00003,42,belum
4,R0_late_roberta_lr3e-05_s42,0,late,roberta,0.00003,42,belum
5,R0_late_distilbert_lr3e-05_s42,0,late,distilbert,0.00003,42,belum


#### Insight
Ada 6 run dan semuanya masih berstatus "belum", artinya folder hasil pilot masih kosong dan tidak tercampur dengan hasil lain.

## 3. Menjalankan Pilot

Keenam run dijalankan satu per satu. Di setiap epoch, model belajar dari data latih lalu dinilai di data validasi. Setelah semua epoch selesai, model dari epoch terbaik dipakai untuk menebak data test dan diukur kecepatannya.

Kalau sel ini dijalankan ulang, run yang sudah selesai akan dilewati.

In [3]:
for cfg in queue:
    T.run_experiment(cfg, DATA_DIR, OUT_DIR, **PILOT)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1317.18it/s]


[R0_comment_roberta_lr3e-05_s42] train 2,000 | val 1,000 | 63 langkah/epoch | parameter 124.6 juta (dilatih 124.6 juta)
  epoch 1 |      7/63 | loss 0.7006 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     13/63 | loss 0.7026 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     19/63 | loss 0.6985 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     26/63 | loss 0.6978 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     32/63 | loss 0.6980 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     38/63 | loss 0.6970 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     45/63 | loss 0.6948 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     51/63 | loss 0.6926 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     57/63 | loss 0.6916 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     63/63 | loss 0.6879 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 0.1 mnt | val loss 0.6497 | acc 0.6370 P 0.6467 R 0.6040 F1 0.6246 macro-F1 0.6366
  epoch 2 |      7/63 | loss 0.6194 | 0.0 mnt, sisa ±0.1 mnt
  epoch 2 |     13/63 | loss 0.6160 | 0.0 mnt, sisa ±0.1 mnt
  epoch 2 |     19/63 | loss 0.6140 | 0

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1269.74it/s]


[R0_comment_distilbert_lr3e-05_s42] train 2,000 | val 1,000 | 63 langkah/epoch | parameter 65.8 juta (dilatih 65.8 juta)
  epoch 1 |      7/63 | loss 0.7105 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     13/63 | loss 0.7002 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     19/63 | loss 0.6953 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     26/63 | loss 0.6942 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     32/63 | loss 0.6927 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     38/63 | loss 0.6894 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     45/63 | loss 0.6860 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     51/63 | loss 0.6831 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     57/63 | loss 0.6795 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     63/63 | loss 0.6780 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 0.0 mnt | val loss 0.6477 | acc 0.6140 P 0.7500 R 0.3420 F1 0.4698 macro-F1 0.5832
  epoch 2 |      7/63 | loss 0.6390 | 0.0 mnt, sisa ±0.0 mnt
  epoch 2 |     13/63 | loss 0.6103 | 0.0 mnt, sisa ±0.0 mnt
  epoch 2 |     19/63 | loss 0.6008 | 

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1030.57it/s]


...

[R0_early_distilbert_lr3e-05_s42] train 2,000 | val 1,000 | 63 langkah/epoch | parameter 65.8 juta (dilatih 65.8 juta)
  epoch 1 |      7/63 | loss 0.6992 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     13/63 | loss 0.6947 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     19/63 | loss 0.6950 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     26/63 | loss 0.6938 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     32/63 | loss 0.6933 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     38/63 | loss 0.6909 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     45/63 | loss 0.6913 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     51/63 | loss 0.6922 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     57/63 | loss 0.6897 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     63/63 | loss 0.6899 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 0.0 mnt | val loss 0.6912 | acc 0.5370 P 0.8364 R 0.0920 F1 0.1658 macro-F1 0.4227
  epoch 2 |      7/63 | loss 0.6861 | 0.0 mnt, sisa ±0.0 mnt
  epoch 2 |     13/63 | loss 0.6620 | 0.0 mnt, sisa ±0.0 mnt
  epoch 2 |     19/63 | loss 0.6547 | 0.

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 881.86it/s]


[R0_late_roberta_lr3e-05_s42] train 2,000 | val 1,000 | 63 langkah/epoch | parameter 127.0 juta (dilatih 127.0 juta)
  epoch 1 |      7/63 | loss 0.7270 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     13/63 | loss 0.7213 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     19/63 | loss 0.7083 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     26/63 | loss 0.7049 | 0.1 mnt, sisa ±0.1 mnt
  epoch 1 |     32/63 | loss 0.7084 | 0.1 mnt, sisa ±0.1 mnt
  epoch 1 |     38/63 | loss 0.7054 | 0.1 mnt, sisa ±0.1 mnt
  epoch 1 |     45/63 | loss 0.7016 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     51/63 | loss 0.7008 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     57/63 | loss 0.7012 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     63/63 | loss 0.7016 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 0.1 mnt | val loss 0.6783 | acc 0.5760 P 0.5739 R 0.5900 F1 0.5819 macro-F1 0.5759
  epoch 2 |      7/63 | loss 0.6799 | 0.0 mnt, sisa ±0.1 mnt
  epoch 2 |     13/63 | loss 0.6913 | 0.0 mnt, sisa ±0.1 mnt
  epoch 2 |     19/63 | loss 0.6870 | 0.0 

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 929.24it/s]


[R0_late_distilbert_lr3e-05_s42] train 2,000 | val 1,000 | 63 langkah/epoch | parameter 67.6 juta (dilatih 67.6 juta)
  epoch 1 |      7/63 | loss 0.7000 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     13/63 | loss 0.7063 | 0.0 mnt, sisa ±0.1 mnt
  epoch 1 |     19/63 | loss 0.7139 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     26/63 | loss 0.7205 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     32/63 | loss 0.7177 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     38/63 | loss 0.7110 | 0.0 mnt, sisa ±0.0 mnt
  epoch 1 |     45/63 | loss 0.7080 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     51/63 | loss 0.7100 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     57/63 | loss 0.7076 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 |     63/63 | loss 0.7082 | 0.1 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 0.1 mnt | val loss 0.6875 | acc 0.5540 P 0.8140 R 0.1400 F1 0.2389 macro-F1 0.4617
  epoch 2 |      7/63 | loss 0.6795 | 0.0 mnt, sisa ±0.1 mnt
  epoch 2 |     13/63 | loss 0.6681 | 0.0 mnt, sisa ±0.1 mnt
  epoch 2 |     19/63 | loss 0.6544 | 0.0

#### Insight
Keenam run selesai tanpa error, masing-masing kurang dari 1 menit. Artinya seluruh bagian training sudah berjalan, mulai dari memuat data sampai menyimpan hasil.

Loss latih turun dari epoch 1 ke epoch 2 di semua run, contohnya RoBERTa comment saja dari 0,6879 menjadi 0,5986. Ini tandanya model memang belajar dari data.

Jumlah parameter RoBERTa 124,6 juta dan DistilBERT 65,8 juta. Model late fusion sedikit lebih besar (127,0 juta dan 67,6 juta) karena lapisan klasifikasinya menerima gabungan empat vektor, bukan satu.

Di epoch 1, beberapa model masih jarang menebak "sarkas". Contohnya DistilBERT early fusion recall-nya hanya 0,092. Ini wajar karena model baru belajar 63 langkah, dan di epoch 2 recall-nya sudah naik menjadi 0,614.

## 4. Hasil Pilot

Tabel pertama berisi ringkasan setiap run (satu baris per run), yaitu nilai validasi di epoch terbaik dan angka efisiensinya. Tabel kedua berisi catatan per epoch (satu baris per epoch), jadi perkembangan model dari epoch ke epoch bisa dilihat.

Keterangan beberapa kolom:
- `peak_mem_gb`: memori GPU tertinggi yang dipakai selama training,
- `latency_bs1_ms`: waktu menebak 1 data kalau data diproses satu per satu,
- `latency_bs32_ms`: waktu menebak per data kalau 32 data diproses sekaligus,
- `train_samples_per_sec`: berapa data yang bisa dilatih per detik.

In [4]:
runs = pd.read_csv(OUT_DIR / "runs.csv")
cols = ["run_id", "best_epoch", "val_accuracy", "val_precision", "val_recall", "val_f1", "val_macro_f1",
        "params_total_m", "peak_mem_gb", "latency_bs1_ms", "latency_bs32_ms"]
display(runs[cols].round(4))

hist = pd.read_csv(OUT_DIR / "history.csv")
display(hist.round(4))

,run_id,best_epoch,val_accuracy,val_precision,val_recall,val_f1,val_macro_f1,params_total_m,peak_mem_gb,latency_bs1_ms,latency_bs32_ms
0,R0_comment_roberta_lr3e-05_s42,2,0.655,0.6807,0.584,0.6286,0.6533,124.6472,3.1866,15.1496,0.5466
1,R0_comment_distilbert_lr3e-05_s42,2,0.640,0.6724,0.546,0.6026,0.6368,65.7830,1.6881,7.6738,0.2940
2,R0_early_roberta_lr3e-05_s42,2,0.635,0.6497,0.586,0.6162,0.6341,124.6472,5.1114,15.7453,1.2446
3,R0_early_distilbert_lr3e-05_s42,2,0.605,0.6031,0.614,0.6085,0.6050,65.7830,2.6457,7.8788,0.6538
4,R0_late_roberta_lr3e-05_s42,1,0.576,0.5739,0.590,0.5819,0.5759,127.0072,6.4437,27.6098,1.5887
5,R0_late_distilbert_lr3e-05_s42,2,0.640,0.6306,0.676,0.6525,0.6395,67.5525,3.3231,13.4541,0.8170


,run_id,epoch,train_loss,val_loss,accuracy,precision,recall,f1,macro_f1,train_min,val_min,train_samples_per_sec,val_samples_per_sec
0,R0_comment_roberta_lr3e-05_s42,1,0.6879,0.6497,0.637,0.6467,0.604,0.6246,0.6366,0.0890,0.0042,374.5186,3944.7156
1,R0_comment_roberta_lr3e-05_s42,2,0.5986,0.6179,0.655,0.6807,0.584,0.6286,0.6533,0.0821,0.0042,406.2182,3950.4721
2,R0_comment_distilbert_lr3e-05_s42,1,0.6780,0.6477,0.614,0.7500,0.342,0.4698,0.5832,0.0430,0.0024,775.6708,7033.5507
3,R0_comment_distilbert_lr3e-05_s42,2,0.5781,0.6205,0.640,0.6724,0.546,0.6026,0.6368,0.0426,0.0024,782.9658,6961.9072
4,R0_early_roberta_lr3e-05_s42,1,0.6948,0.6788,0.597,0.7118,0.326,0.4472,0.5651,0.0901,0.0080,369.9851,2075.1542
5,R0_early_roberta_lr3e-05_s42,2,0.6463,0.6413,0.635,0.6497,0.586,0.6162,0.6341,0.0947,0.0078,352.0364,2132.0580
6,R0_early_distilbert_lr3e-05_s42,1,0.6899,0.6912,0.537,0.8364,0.092,0.1658,0.4227,0.0484,0.0044,688.9049,3761.5423
7,R0_early_distilbert_lr3e-05_s42,2,0.6324,0.6541,0.605,0.6031,0.614,0.6085,0.6050,0.0484,0.0044,688.3036,3792.7346
8,R0_late_roberta_lr3e-05_s42,1,0.7016,0.6783,0.576,0.5739,0.590,0.5819,0.5759,0.1346,0.0129,247.6007,1291.6557
9,R0_late_roberta_lr3e-05_s42,2,0.6659,0.6419,0.613,0.6638,0.458,0.5420,0.6035,0.1335,0.0122,249.7001,1369.9719


#### Insight
Nilai F1 di pilot masih rendah, sekitar 0,58 sampai 0,65, dan naik turunnya belum teratur. Ini wajar karena model hanya belajar dari 2.000 data. Jadi angka performa di sini belum bisa dipakai untuk membandingkan model atau jenis input.

Angka efisiensi sudah bisa memberi gambaran awal:
- DistilBERT sekitar 2 kali lebih cepat dari RoBERTa di ketiga input. Contohnya pada early fusion, DistilBERT melatih 688 data per detik, sedangkan RoBERTa 352 data per detik. Memorinya juga sekitar setengah (2,65 GB dibanding 5,11 GB).
- Late fusion ternyata lebih lambat dan lebih boros daripada early fusion. Pada RoBERTa, kecepatan training late fusion 250 data per detik, sedangkan early fusion 352 data per detik. Waktu menebak 1 data juga hampir dua kali lebih lama (27,6 ms dibanding 15,7 ms). Penyebabnya, late fusion menjalankan model dua kali, sekali untuk parent dan sekali untuk comment.
- Comment saja hanya sedikit lebih cepat dari early fusion (406 dibanding 352 data per detik pada RoBERTa), padahal teksnya hampir tiga kali lebih pendek. Untuk teks yang pendek, waktu training lebih banyak dipengaruhi oleh jumlah langkah daripada panjang teksnya.
- Memori tertinggi hanya 6,44 GB (RoBERTa late fusion), jadi aman untuk GPU 16 GB.

Ada satu hal yang ditemukan dari pilot. Pada RoBERTa late fusion, epoch 1 terpilih sebagai yang terbaik karena F1-nya lebih tinggi (0,5819 dibanding 0,5420). Padahal accuracy dan macro-F1 epoch 2 lebih baik (accuracy 0,613 dibanding 0,576). Ini terjadi karena F1 kelas sarkas cenderung memilih model yang lebih sering menebak "sarkas". Karena itu untuk training data penuh, epoch terbaik dipilih memakai macro-F1, yaitu rata-rata F1 kelas sarkas dan kelas bukan sarkas. Accuracy, precision, recall, dan F1 tetap dicatat semuanya.

## 5. Perkiraan Waktu Training Data Penuh

Waktu training data penuh diperkirakan dari kecepatan di epoch 2 pilot. Caranya, jumlah data penuh dibagi dengan jumlah data yang bisa diproses per detik. Hasilnya dijumlahkan untuk 3 epoch, ditambah waktu menilai data validasi setiap epoch dan waktu menebak data test.

In [5]:
N_TRAIN, N_VAL, N_TEST, EPOCHS = 799_459, 100_276, 101_129, 3

last = hist.sort_values("epoch").groupby("run_id").tail(1)          # epoch ke-2 (GPU sudah "panas")
est = last.merge(runs[["run_id", "input", "model", "test_pred_min", "n_test"]], on="run_id")
est["train_mnt_per_epoch"] = N_TRAIN / est["train_samples_per_sec"] / 60
est["val_mnt_per_epoch"] = N_VAL / est["val_samples_per_sec"] / 60
est["test_mnt"] = est["test_pred_min"] * N_TEST / est["n_test"]
est["total_jam"] = (EPOCHS * (est["train_mnt_per_epoch"] + est["val_mnt_per_epoch"]) + est["test_mnt"]) / 60

display(est[["input", "model", "train_mnt_per_epoch", "val_mnt_per_epoch", "test_mnt", "total_jam"]].round(2))
print(f"Perkiraan total putaran 1 (6 run x {EPOCHS} epoch): {est['total_jam'].sum():.1f} jam")

,input,model,train_mnt_per_epoch,val_mnt_per_epoch,test_mnt,total_jam
0,comment,distilbert,17.02,0.24,0.25,0.87
1,comment,roberta,32.80,0.42,0.40,1.67
2,early,distilbert,19.36,0.44,0.46,1.00
3,early,roberta,37.85,0.78,0.84,1.95
4,late,roberta,53.36,1.22,1.29,2.75
5,late,distilbert,27.81,0.69,0.71,1.44


Perkiraan total putaran 1 (6 run x 3 epoch): 9.7 jam


#### Insight
Perkiraan waktu satu putaran (6 run, masing-masing 3 epoch) adalah sekitar 9,7 jam. Run paling lama adalah RoBERTa late fusion (sekitar 2,75 jam), dan yang paling cepat DistilBERT comment saja (sekitar 0,87 jam).

Hampir semua waktu dipakai untuk training. Menilai data validasi dan menebak data test hanya butuh kurang dari 1,5 menit per epoch.

Angka ini masih perkiraan dari 2.000 data, jadi waktu sebenarnya bisa sedikit berbeda. Waktu yang sebenarnya akan terlihat setelah run pertama dengan data penuh selesai.

## 6. Kesimpulan

1. Seluruh proses training sudah berjalan tanpa error untuk ketiga jenis input dan kedua model.
2. Nilai performa di pilot belum bisa dibandingkan karena datanya terlalu sedikit.
3. Dari sisi efisiensi, DistilBERT sekitar 2 kali lebih cepat dan hemat memori dibanding RoBERTa, sedangkan late fusion lebih lambat dibanding early fusion.
4. Training data penuh untuk 6 run diperkirakan sekitar 9,7 jam.
5. Untuk training data penuh, epoch terbaik dipilih memakai macro-F1.